## Transaction-Level Attribution Rule

유니크 총매출을 구하기 위해 각 transaction을 오직 하나의 offer에만 귀속시키는 attribution 규칙을 적용한다.

### 기본 원칙
- `offer completed`가 존재하는 offer만 귀속 후보로 본다.
- `transaction time`과 `offer completed time`이 같으면 해당 offer에 우선 귀속한다.
- 그 외에는 transaction 시점에 활성화된 `received ~ completed` window들 중
  가장 최근에 `received`된 offer에 귀속한다.  
  (last-touch + completed boundary priority)

### 귀속 우선순위
1. `transaction time == offer completed time`
2. transaction 시점에 포함되는 candidate offer들 중 가장 최근 `received`된 offer
3. 후보가 여러 개면 추가 tie-break 규칙으로 정렬
4. 후보가 없으면 `unmatched`로 둔다

### 왜 이 규칙을 쓰는가
- 기존의 received ~ completed 누적 방식은 여러 offer window가 겹칠 때 동일 transaction이 여러 offer에 중복 귀속될 수 있다.
- transaction-level attribution은 이런 중복을 제거하고, 유니크한 총매출을 계산하기 위한 방식이다.
- 모든 completed offer에 동일한 규칙을 적용해야 총합이 일관된다.

### 주의점
- 이 방식은 실제 인과를 완벽히 복원하는 것이 아니라,
  명시한 규칙에 따라 transaction을 배분하는 attribution 방식이다.
- 따라서 결과는 "유니크 총매출"에 가까운 해석을 제공하지만,
  규칙 정의에 따라 수치가 달라질 수 있다.


### 각 transaction으로 발생한 amount가 어떤 offer의 영향을 받은 건지 확인

In [5]:
# 1. 불러오기

import pandas as pd

# 최종 보정이 반영된 쿠폰 테이블
promotion_final = pd.read_csv('./data/promotion_final.csv')

# 모든 transaction이 들어 있는 원본 테이블
merged = pd.read_csv('./data/all_merged.csv')


In [6]:
# 2. 완료된 쿠폰 window 만들기

# transaction 귀속 대상이 되는 completed offer만 뽑는다
# is_deduplicated = 1
# is_completed = 1
windows = promotion_final[
    (promotion_final['is_deduplicated'].eq(1)) &
    (promotion_final['is_completed'].eq(1)) &
    (promotion_final['offer received'].notna()) &
    (promotion_final['offer completed'].notna())
].copy()

# 귀속된 금액을 저장할 칸을 만든다
windows['attributed_amount'] = 0.0

# viewed 이후 금액을 따로 저장할 칸도 만든다
windows['unique_promo_influenced_amount'] = 0.0

# 나중에 보기 편하게 정렬
windows = windows.sort_values(['person', 'offer received', 'offer completed', 'offer_cycle']).reset_index(drop=True)

# 같은 사람의 window를 빠르게 찾기 위한 묶음
window_groups = {
    person: g.copy()
    for person, g in windows.groupby('person', sort=False)
}

# transaction만 따로 뽑는다
transactions = merged[merged['event'].eq('transaction')].copy()
transactions = transactions[['person', 'time', 'amount']].sort_values(['person', 'time']).reset_index(drop=True)


In [7]:
def choose_window(active_windows, tx_time):
    # 후보가 없으면 귀속할 수 없음
    if len(active_windows) == 0:
        return None

    # 우선순위:
    # 1) transaction 시각과 completed 시각이 정확히 같은 window
    # 2) 가장 최근 received
    # 3) 가장 최근 completed
    # 4) offer_cycle는 마지막 정렬용
    active_windows = active_windows.sort_values(
        ['completed_match', 'offer received', 'offer completed', 'offer_cycle'],
        ascending=[False, False, False, False]
    )
    return active_windows.iloc[0]

matched = 0
unmatched = 0

for person, tx_group in transactions.groupby('person', sort=False):
    person_windows = window_groups.get(person)

    # 이 사람에게 completed window가 하나도 없으면 전부 unmatched
    if person_windows is None or len(person_windows) == 0:
        unmatched += len(tx_group)
        continue

    # window를 하나씩 앞으로 열어가며 active 후보를 만든다
    open_idx = 0
    active_idx = []

    # transaction을 시간순으로 처리
    for _, tx in tx_group.iterrows():
        tx_time = tx['time']
        tx_amount = tx['amount']

        # 현재 시점까지 시작된 window를 active에 넣는다
        while open_idx < len(person_windows) and person_windows.iloc[open_idx]['offer received'] <= tx_time:
            active_idx.append(person_windows.index[open_idx])
            open_idx += 1

        # 아직 끝나지 않은 window만 남긴다
        active_idx = [
            idx for idx in active_idx
            if person_windows.loc[idx, 'offer completed'] >= tx_time
        ]

        # active window가 없으면 unmatched
        if len(active_idx) == 0:
            unmatched += 1
            continue

        # 후보 window들만 따로 놓고 우선순위 판단
        candidate_windows = person_windows.loc[active_idx, [
            'person', 'offer_id', 'offer_cycle',
            'offer received', 'offer viewed', 'offer completed'
        ]].copy()

        # exact completed-time match 여부 표시
        candidate_windows['completed_match'] = (
            candidate_windows['offer completed'].eq(tx_time)
        )

        chosen = choose_window(candidate_windows, tx_time)
        if chosen is None:
            unmatched += 1
            continue

        # 선택된 window의 원래 인덱스를 찾아서 금액을 더한다
        chosen_mask = (
            (person_windows['offer_id'].eq(chosen['offer_id'])) &
            (person_windows['offer_cycle'].eq(chosen['offer_cycle']))
        )
        chosen_idx = person_windows[chosen_mask].index[0]

        # 전체 귀속 금액
        windows.loc[chosen_idx, 'attributed_amount'] += tx_amount

        # viewed 이후 금액
        viewed_time = windows.loc[chosen_idx, 'offer viewed']
        completed_time = windows.loc[chosen_idx, 'offer completed']

        if pd.notna(viewed_time):
            if tx_time >= viewed_time and tx_time <= completed_time:
                windows.loc[chosen_idx, 'unique_promo_influenced_amount'] += tx_amount

        matched += 1

print('matched transactions:', matched)
print('unmatched transactions:', unmatched)


matched transactions: 32604
unmatched transactions: 106349


In [8]:
# 확인용 셀
offer_summary = windows.copy()

# reward를 뺀 순기여 proxy
offer_summary['net_profit_proxy'] = offer_summary['attributed_amount'] - offer_summary['reward']

# reward 대비 효율
offer_summary['profit_per_reward'] = offer_summary['net_profit_proxy'] / offer_summary['reward']

display(
    offer_summary[
        [
            'person', 'offer_id', 'offer_cycle',
            'offer_type', 'difficulty', 'reward',
            'offer received', 'offer viewed', 'offer completed',
            'amount', 'attributed_amount',
            'unique_promo_influenced_amount',
            'net_profit_proxy', 'profit_per_reward'
        ]
    ].head(20)
)



,person,offer_id,offer_cycle,offer_type,difficulty,reward,offer received,offer viewed,offer completed,amount,attributed_amount,unique_promo_influenced_amount,net_profit_proxy,profit_per_reward
0,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,discount,7,3,168.0,186.0,252.0,11.93,11.93,11.93,8.93,2.976667
1,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,bogo,5,5,504.0,516.0,576.0,22.05,22.05,22.05,17.05,3.410000
2,0020c2b971eb4e9188eac86d93036a77,discount_10_2_10,Discount_1,discount,10,2,0.0,12.0,54.0,17.63,17.63,17.63,15.63,7.815000
3,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_5,Bogo_1,bogo,10,10,408.0,426.0,510.0,17.24,17.24,17.24,7.24,0.724000
4,0020ccbbb6d84e358d3414a3ff76cffd,discount_7_3_7,Discount_1,discount,7,3,168.0,168.0,222.0,11.65,11.65,11.65,8.65,2.883333
5,0020ccbbb6d84e358d3414a3ff76cffd,bogo_5_5_5,Bogo_1,bogo,5,5,336.0,348.0,378.0,14.53,14.53,14.53,9.53,1.906000
6,0020ccbbb6d84e358d3414a3ff76cffd,bogo_5_5_7,Bogo_1,bogo,5,5,504.0,582.0,600.0,10.32,10.32,10.32,5.32,1.064000
7,003d66b6608740288d6cc97a6903f4f0,discount_10_2_10,Discount_1,discount,10,2,168.0,300.0,384.0,10.51,10.51,9.03,8.51,4.255000
8,003d66b6608740288d6cc97a6903f4f0,discount_10_2_10,Discount_2,discount,10,2,408.0,420.0,504.0,13.44,13.44,13.44,11.44,5.720000
9,00426fe3ffde4c6b9cb9ad6d077a13ea,discount_10_2_10,Discount_1,discount,10,2,168.0,186.0,258.0,23.34,23.34,23.34,21.34,10.670000


In [10]:
# normal flow
normal_rows = offer_summary[offer_summary['is_normal_flow'].eq(1)].copy()
normal_current_total = normal_rows['amount'].sum()
normal_attributed_total = normal_rows['attributed_amount'].sum()
normal_promo_total = normal_rows['unique_promo_influenced_amount'].sum()

print('normal flow rows:', len(normal_rows))
print('normal current total:', round(normal_current_total, 2))
print('normal attributed total:', round(normal_attributed_total, 2))
print('normal promo influenced total:', round(normal_promo_total, 2))
print('normal diff:', round(normal_attributed_total - normal_current_total, 2))

# bogo
bogo_rows = offer_summary[offer_summary['offer_type'].eq('bogo')].copy()
bogo_current_total = bogo_rows['amount'].sum()
bogo_attributed_total = bogo_rows['attributed_amount'].sum()
bogo_promo_total = bogo_rows['unique_promo_influenced_amount'].sum()

print('bogo rows:', len(bogo_rows))
print('bogo current total:', round(bogo_current_total, 2))
print('bogo attributed total:', round(bogo_attributed_total, 2))
print('bogo promo influenced total:', round(bogo_promo_total, 2))
print('bogo diff:', round(bogo_attributed_total - bogo_current_total, 2))

# discount
discount_rows = offer_summary[offer_summary['offer_type'].eq('discount')].copy()
discount_current_total = discount_rows['amount'].sum()
discount_attributed_total = discount_rows['attributed_amount'].sum()
discount_promo_total = discount_rows['unique_promo_influenced_amount'].sum()

print('discount rows:', len(discount_rows))
print('discount current total:', round(discount_current_total, 2))
print('discount attributed total:', round(discount_attributed_total, 2))
print('discount promo influenced total:', round(discount_promo_total, 2))
print('discount diff:', round(discount_attributed_total - discount_current_total, 2))


normal flow rows: 22333
normal current total: 486645.59
normal attributed total: 484301.34
normal promo influenced total: 477292.38
normal diff: -2344.25
bogo rows: 10600
bogo current total: 235413.54
bogo attributed total: 235158.25
bogo promo influenced total: 233704.35
bogo diff: -255.29
discount rows: 11733
discount current total: 251232.05
discount attributed total: 249143.09
discount promo influenced total: 243588.03
discount diff: -2088.96


In [12]:
offer_summary.to_csv('./data/offer_summary.csv', index=False)


In [13]:
offer_summary_kr = offer_summary.rename(columns={
    'person': '회원ID',
    'offer_id': '오퍼ID',
    'offer_cycle': '오퍼회차',
    'offer_type': '오퍼유형',
    'difficulty': '목표금액',
    'reward': '보상금액',
    'duration': '기간',
    'offer received': '수신시각',
    'offer viewed': '열람시각',
    'offer completed': '완료시각',
    'amount': '원본금액',
    'attributed_amount': '귀속금액',
    'unique_promo_influenced_amount': '유니크프로모션영향액',
    'net_profit_proxy': '순이익대용값',
    'profit_per_reward': '보상대비효율',
    'is_normal_flow': '정상흐름여부',
    'is_deduplicated': '중복제거여부'
})

display(
    offer_summary_kr[
        [
            '회원ID', '오퍼ID', '오퍼회차', '오퍼유형',
            '목표금액', '보상금액', '기간',
            '수신시각', '열람시각', '완료시각',
            '원본금액', '귀속금액', '유니크프로모션영향액',
            '순이익대용값', '보상대비효율',
            '정상흐름여부', '중복제거여부'
        ]
    ].head(20)
)


,회원ID,오퍼ID,오퍼회차,오퍼유형,목표금액,보상금액,기간,수신시각,열람시각,완료시각,원본금액,귀속금액,유니크프로모션영향액,순이익대용값,보상대비효율,정상흐름여부,중복제거여부
0,0011e0d4e6b944f998e987f904e8c1e5,discount_7_3_7,Discount_1,discount,7,3,7,168.0,186.0,252.0,11.93,11.93,11.93,8.93,2.976667,1,1
1,0011e0d4e6b944f998e987f904e8c1e5,bogo_5_5_7,Bogo_1,bogo,5,5,7,504.0,516.0,576.0,22.05,22.05,22.05,17.05,3.410000,1,1
2,0020c2b971eb4e9188eac86d93036a77,discount_10_2_10,Discount_1,discount,10,2,10,0.0,12.0,54.0,17.63,17.63,17.63,15.63,7.815000,1,1
3,0020c2b971eb4e9188eac86d93036a77,bogo_10_10_5,Bogo_1,bogo,10,10,5,408.0,426.0,510.0,17.24,17.24,17.24,7.24,0.724000,1,1
4,0020ccbbb6d84e358d3414a3ff76cffd,discount_7_3_7,Discount_1,discount,7,3,7,168.0,168.0,222.0,11.65,11.65,11.65,8.65,2.883333,1,1
5,0020ccbbb6d84e358d3414a3ff76cffd,bogo_5_5_5,Bogo_1,bogo,5,5,5,336.0,348.0,378.0,14.53,14.53,14.53,9.53,1.906000,1,1
6,0020ccbbb6d84e358d3414a3ff76cffd,bogo_5_5_7,Bogo_1,bogo,5,5,7,504.0,582.0,600.0,10.32,10.32,10.32,5.32,1.064000,1,1
7,003d66b6608740288d6cc97a6903f4f0,discount_10_2_10,Discount_1,discount,10,2,10,168.0,300.0,384.0,10.51,10.51,9.03,8.51,4.255000,1,1
8,003d66b6608740288d6cc97a6903f4f0,discount_10_2_10,Discount_2,discount,10,2,10,408.0,420.0,504.0,13.44,13.44,13.44,11.44,5.720000,1,1
9,00426fe3ffde4c6b9cb9ad6d077a13ea,discount_10_2_10,Discount_1,discount,10,2,10,168.0,186.0,258.0,23.34,23.34,23.34,21.34,10.670000,1,1


원본금액(amount)은 기존 방식의 금액 : 05_UK에서 로직을 적용한 금액 -> 누적값이 있는 offer를 received와 completed 까지 time의 amount를 더하는 로직(필요시 한 회차 이전의 received까지 확인해서 더함)

귀속금액(attributed_amount)은 unique attribution으로 다시 계산한 귀속 금액 -> 한 amount가 하나의 offer에만 영향을 주도록 함

유니크 프로모션 영향액(unique_promo_influenced_amount)은 viewed 이후의 유니크 프로모션 영향액

순이익대용값(net_profit_proxy)은 귀속금액에서 reward를 뺀 값

In [ ]:
############################################################

# Transaction-Level Attribution 정리

이 노트북에서는 `promotion_final.csv`와 `all_merged.csv`를 이용해 transaction을 offer에 하나씩 귀속시키는 작업을 했다.

## 작업 내용
- `promotion_final.csv`에서 완료된 offer만 window로 만들었다.
- `all_merged.csv`에서 transaction만 따로 뽑았다.
- 각 transaction을 시각 기준으로 하나의 offer window에만 귀속시켰다.
- 귀속 우선순위는 `transaction time == offer completed time`을 먼저 보고, 그다음 가장 최근 `offer received`를 보는 방식으로 정했다.
- 그 결과를 `attributed_amount`에 누적했다.
- `offer viewed` 이후에 발생한 금액만 따로 더해 `unique_promo_influenced_amount`도 계산했다.
- normal, bogo, discount 각각에 대해 기존 합계와 unique attribution 후 합계를 비교했다.
- 마지막에 offer별 요약 테이블인 `offer_summary`를 만들었다.

## 핵심 정리
- `attributed_amount` = transaction-level attribution으로 귀속된 유니크 매출
- `unique_promo_influenced_amount` = 그중 viewed 이후 매출
- `net_profit_proxy` = `attributed_amount - reward`
- `profit_per_reward` = reward 대비 효율

## 해석
- 기존 `amount`는 window가 겹칠 때 같은 transaction이 여러 offer에 중복 귀속될 수 있었다.
- transaction-level attribution을 적용하면 transaction 하나를 하나의 offer에만 붙여서 중복을 줄일 수 있다.
- 그 결과 offer별 수익성과 프로모션 영향액을 더 일관되게 볼 수 있다.


In [ ]:
############################################################

# 최종 정리

이번 작업은 크게 두 단계였다.

## 1. `05_UK_difficulty_check.ipynb`
- offer row 기준으로 amount를 다시 계산했다.
- current 기준으로 부족한 행만 prev_received로 보정했다.
- viewed 이후 프로모션 영향액도 함께 정리했다.
- 최종 `promotion_final.csv`를 만들었다.

## 2. `05__2_UK_결제금액_구하기.ipynb`
- transaction을 offer window 하나에만 귀속시켰다.
- 중복 없는 유니크 매출인 `attributed_amount`를 계산했다.
- `unique_promo_influenced_amount`도 함께 계산했다.
- offer별 성과와 수익성까지 볼 수 있는 `offer_summary`를 만들었다.

## 전체 요약
- 첫 번째 노트북은 offer row의 amount를 정리한 작업이다.
- 두 번째 노트북은 transaction-level attribution으로 유니크 매출과 프로모션 영향액을 구한 작업이다.
- 이제 offer별로 매출, 영향액, reward 대비 효율까지 함께 볼 수 있다.
